In [6]:
import json
import pandas as pd
from pathlib import Path

# 1. Load the raw JSON (list of nested dicts)
data_path = Path("../data/raw_credit_applications.json")  # or the correct relative/path in your repo
with open(data_path, "r") as f:
    raw_data = json.load(f)

# Quick sanity check
len(raw_data), type(raw_data[0])

# 2. Normalize nested JSON into a flat table
df = pd.json_normalize(raw_data)

# 3. Inspect and save
df.info()

# Basic data-quality overview
print("\nMissing values per column:")
print(df.isna().sum())

# Save for team
output_path = Path("../data/normalized_credit_applications.csv")
df.to_csv(output_path, index=False)
output_path


<class 'pandas.DataFrame'>
RangeIndex: 502 entries, 0 to 501
Data columns (total 21 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   _id                               502 non-null    str    
 1   spending_behavior                 502 non-null    object 
 2   processing_timestamp              62 non-null     str    
 3   applicant_info.full_name          502 non-null    str    
 4   applicant_info.email              502 non-null    str    
 5   applicant_info.ssn                497 non-null    str    
 6   applicant_info.ip_address         497 non-null    str    
 7   applicant_info.gender             501 non-null    str    
 8   applicant_info.date_of_birth      501 non-null    str    
 9   applicant_info.zip_code           501 non-null    str    
 10  financials.annual_income          497 non-null    object 
 11  financials.credit_history_months  502 non-null    int64  
 12  financials.debt_to_

,_id,spending_behavior,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,...,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes
0,app_200,"[{'category': 'Shopping', 'amount': 480}, {'ca...",2024-01-15T00:00:00Z,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036,...,23,0.20,31212,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN
1,app_037,"[{'category': 'Rent', 'amount': 608}, {'catego...",NaN,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,M,1992-03-31,10032,...,51,0.18,17915,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN
2,app_215,"[{'category': 'Rent', 'amount': 109}]",NaN,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075,...,41,0.21,37909,True,NaN,vacation,3.7,59000.0,NaN,NaN
3,app_024,"[{'category': 'Fitness', 'amount': 575}]",NaN,Thomas Lee,thomas.lee6@protonmail.com,194-35-1833,192.168.175.67,Male,1983-04-25,10077,...,70,0.35,0,True,NaN,NaN,4.3,34000.0,NaN,NaN
4,app_184,"[{'category': 'Entertainment', 'amount': 463}]",2024-01-15T00:00:00Z,Brian Rodriguez,brian.rodriguez86@aol.com,480-41-2475,172.29.125.105,M,1999-05-21,10080,...,14,0.23,31763,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
497,app_468,"[{'category': 'Transportation', 'amount': 701}]",NaN,Patrick Martinez,patrick.martinez26@aol.com,996-86-6099,172.31.95.30,Male,1999-05-23,10046,...,35,0.35,8982,False,insufficient_credit_history,NaN,NaN,NaN,NaN,NaN
498,app_192,"[{'category': 'Healthcare', 'amount': 650}]",2024-01-15T00:00:00Z,Dennis Lopez,dennis.lopez78@yahoo.com,936-44-8002,172.29.183.18,Male,11/15/1985,10088,...,40,0.22,34292,True,NaN,NaN,6.0,18000.0,NaN,NaN
499,app_234,"[{'category': 'Insurance', 'amount': 526}]",NaN,Samuel Hill,samuel.hill67@protonmail.com,937-72-8731,10.143.146.157,Male,1976/01/29,10090,...,60,0.30,38703,False,algorithm_risk_score,education,NaN,NaN,NaN,NaN
500,app_306,"[{'category': 'Insurance', 'amount': 490}]",NaN,Anna White,anna.white6@gmail.com,757-27-8131,10.26.176.56,Female,1978-07-26,90227,...,80,0.29,63560,True,NaN,NaN,3.1,57000.0,NaN,NaN


In [7]:
# 1. Overview of missing data
missing_counts = df.isna().sum().sort_values(ascending=False)
missing_percent = (missing_counts / len(df) * 100).round(1)

missing_summary = pd.DataFrame({
    "missing_count": missing_counts,
    "missing_percent": missing_percent
})

missing_summary


,missing_count,missing_percent
notes,500,99.6
financials.annual_salary,497,99.0
loan_purpose,452,90.0
processing_timestamp,440,87.6
decision.rejection_reason,292,58.2
decision.approved_amount,210,41.8
decision.interest_rate,210,41.8
financials.annual_income,5,1.0
applicant_info.ip_address,5,1.0
applicant_info.ssn,5,1.0


In [8]:
# 2. Check for duplicate application IDs
total_rows = df.shape[0]
unique_ids = df["_id"].nunique()
duplicate_rows = total_rows - unique_ids

print("Total rows:", total_rows)
print("Unique _id values:", unique_ids)
print("Duplicate rows (by _id):", duplicate_rows)

# 3. Create cleaned copy and drop duplicates
df_clean = df.copy()

before = df_clean.shape[0]
df_clean = df_clean.drop_duplicates(subset="_id", keep="first")
after = df_clean.shape[0]

print("Rows before dropping duplicates:", before)
print("Rows after dropping duplicates:", after)
print("Rows removed:", before - after)


Total rows: 502
Unique _id values: 500
Duplicate rows (by _id): 2
Rows before dropping duplicates: 502
Rows after dropping duplicates: 500
Rows removed: 2


In [9]:
# 4. Ensure numeric columns are numeric in df_clean
numeric_cols = [
    "financials.annual_income",
    "financials.credit_history_months",
    "financials.debt_to_income",
    "financials.savings_balance",
    "decision.interest_rate",
    "decision.approved_amount",
]

for col in numeric_cols:
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

df_clean[numeric_cols].dtypes


financials.annual_income            float64
financials.credit_history_months      int64
financials.debt_to_income           float64
financials.savings_balance            int64
decision.interest_rate              float64
decision.approved_amount            float64
dtype: object

In [10]:
# 5. Simple sanity checks on numeric values in df_clean

print("Negative or zero income:",
      df_clean[df_clean["financials.annual_income"] <= 0].shape[0])

print("Very high income (> 1,000,000):",
      df_clean[df_clean["financials.annual_income"] > 1_000_000].shape[0])

print("Negative savings:",
      df_clean[df_clean["financials.savings_balance"] < 0].shape[0])

print("Weird debt_to_income (<0 or >5):",
      df_clean[
          (df_clean["financials.debt_to_income"] < 0) |
          (df_clean["financials.debt_to_income"] > 5)
      ].shape[0])
# “We found 1 application with non‑positive annual income, which is invalid for a credit application.”

# “We found 1 application with negative savings balance.”

# “We did not find extremely high incomes above 1,000,000 or implausible debt‑to‑income ratios outside.”

Negative or zero income: 1
Very high income (> 1,000,000): 0
Negative savings: 1
Weird debt_to_income (<0 or >5): 0


In [11]:
from pathlib import Path

clean_path = Path("../data/clean_credit_applications.csv")
df_clean.to_csv(clean_path, index=False)
clean_path


PosixPath('../data/clean_credit_applications.csv')

In [12]:
# Drop columns with >50% missing values
threshold = 0.5  # 50%
cols_to_drop = missing_percent[missing_percent > threshold].index.tolist()
print("Dropping columns with >50% missing:", cols_to_drop)

df_analysis = df_clean.drop(columns=cols_to_drop)
print("Remaining columns:", df_analysis.shape[1])


Dropping columns with >50% missing: ['notes', 'financials.annual_salary', 'loan_purpose', 'processing_timestamp', 'decision.rejection_reason', 'decision.approved_amount', 'decision.interest_rate', 'financials.annual_income', 'applicant_info.ip_address', 'applicant_info.ssn']
Remaining columns: 11


In [14]:
# Simple imputation for bias analysis (only key columns for bias work)
df_analysis["applicant_info.gender"] = df_analysis["applicant_info.gender"].fillna("Unknown")

print("Gender missing before:", df_clean["applicant_info.gender"].isna().sum())
print("Gender missing after:", df_analysis["applicant_info.gender"].isna().sum())


Gender missing before: 0
Gender missing after: 0


In [18]:
from pathlib import Path
final_path = Path("../data/analysis_ready_credit_applications.csv")
df_analysis.to_csv(final_path, index=False)
print("FINAL analysis-ready dataset saved!")
print("Shape:", df_analysis.shape)


FINAL analysis-ready dataset saved!
Shape: (500, 11)


In [19]:
import os
print(os.getcwd())

/Users/magdalenakrzhovska/dego_gp_txc1/notebooks
